# Check raw MMN Wilson loops

Compares the raw occupied MMN geometry of Ti_Q_2A trials 03 and 04. The two checkpoints use different Wannier spaces (48 and 38 functions), but their 38-dimensional occupied subspaces should coincide on the original $8^3$ first-principles mesh. This deliberately does **not** construct $A(k)$, $A(R)$, $H(R)$, derivatives, or any off-grid Fourier interpolant -- it establishes what trial-to-trial differences downstream *cannot* be blamed on.

At each mesh point, $C_a(k)$ is an orthonormal basis for the physical occupied subspace in the common 100-band ab-initio basis. For every raw MMN edge $k \to k+b$ it constructs
$$L_a(k,b) = C_a(k)^\dagger M(k,b) C_a(k+b).$$
The link matrices are basis covariant rather than invariant, so they're aligned with
$$Q(k) = \mathrm{polar}[C_3(k)^\dagger C_4(k)], \qquad L_4(k,b) \to Q(k)^\dagger L_3(k,b)\, Q(k+b).$$
For each positive-oriented $(\mu,\nu)$ plaquette on the periodic $8^3$ torus, it constructs the polar-unitary Wilson loop $W_{\mu\nu}(k) = U_\mu(k)\,U_\nu(k+\mu)\,U_\mu(k+\nu)^\dagger\,U_\nu(k)^\dagger$, where $U=\mathrm{polar}(L)$. $\arg\det W$ is compared directly between trials for xy, yz, and zx. Raw (not polar-unitarized) loops and loops using the actual negative-direction MMN records are also checked as diagnostics.

The multi-GB MMN is streamed record-by-record. Existing `.mmn`, `.mmn.gz`, `.chk`, and `.chk.gz` inputs are opened read-only and are never modified. `RUN_COMPUTATION = False` loads the last saved `results/raw_mmn_wilson/` instead of re-streaming the MMN (this is the slowest check in `inputs/`, `nntot` neighbours times `num_kpts` records).

In [ ]:
import sys
import csv
import json
from pathlib import Path

import numpy as np

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "validation").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from validation._paths import (
    RESULTS,
    STO_TI_Q_2A_RUN,
)
from validation._paths import DATA
from wannier.wannier_io import (
    _mmn_header,
    _mmn_records,
    check_mmn_compatibility,
    read_wannier_checkpoint,
)

## Config

In [ ]:
RUN_COMPUTATION = False  # True: stream the full MMN and recompute (slow). False: load results/raw_mmn_wilson/.
PREFIX = "SrTiO3"
N_OCC = 38
RUN = STO_TI_Q_2A_RUN
TRIAL03 = RUN / "trial_03_Sr_sp_Ti_pd_O_sp/proj_gauge"
TRIAL04 = DATA / "SrTiO3/Ti_Q_2A/output/188914bc889/trial_04_Sr_sp_Ti_p_O_sp/proj_gauge/188927bc889"
MMN_PATH = TRIAL03 / f"{PREFIX}.mmn"
OUTPUT_DIR = RESULTS / "raw_mmn_wilson"  # was --output

# The ordering keeps the usual Cartesian curvature labels.
PLAQUETTES = ((0, 1, "xy"), (1, 2, "yz"), (2, 0, "zx"))

## Small helpers

In [ ]:
def polar_unitary(matrix: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Return the unitary polar factor and singular values of a square matrix."""
    left, singular_values, right_h = np.linalg.svd(matrix, full_matrices=False)
    return left @ right_h, singular_values


def wrapped_phase_difference(left: float, right: float) -> float:
    """Return ``right-left`` wrapped onto (-pi,pi]."""
    return float(np.angle(np.exp(1j * (right - left))))


def determinant_phase(matrix: np.ndarray) -> float:
    """Return arg(det(matrix)) without using the determinant magnitude."""
    phase, _ = np.linalg.slogdet(matrix)
    return float(np.angle(phase))


def read_eigenvalues(path: Path, num_kpts: int, num_bands: int) -> np.ndarray:
    """Read seedname.eig into ``(k, band)`` ordering, checking completeness."""
    raw = np.loadtxt(path)
    if raw.ndim != 2 or raw.shape[1] != 3:
        raise ValueError(f"{path}: expected three columns")
    values = np.full((num_kpts, num_bands), np.nan, dtype=float)
    band = raw[:, 0].astype(int) - 1
    kpoint = raw[:, 1].astype(int) - 1
    if np.any(band < 0) or np.any(band >= num_bands) or np.any(kpoint < 0) or np.any(kpoint >= num_kpts):
        raise ValueError(f"{path}: band or k-point index is out of range")
    values[kpoint, band] = raw[:, 2]
    if not np.isfinite(values).all():
        raise ValueError(f"{path}: eigenvalue table is incomplete")
    return values


def validate_inputs(checkpoint03: dict, checkpoint04: dict) -> tuple[int, ...]:
    """Check that both checkpoints and the shared MMN describe one mesh."""
    check_mmn_compatibility(TRIAL04, TRIAL03, PREFIX)
    for key in ("num_bands", "num_kpts", "nntot", "mp_grid"):
        if checkpoint03[key] != checkpoint04[key]:
            raise ValueError(f"checkpoint mismatch: {key} is {checkpoint03[key]} versus {checkpoint04[key]}")
    if int(checkpoint03["num_wann"]) != 48:
        raise ValueError("trial 03 is not the expected 48-WF checkpoint")
    if not np.allclose(checkpoint03["kpt_red"], checkpoint04["kpt_red"], atol=1e-13):
        raise ValueError("checkpoint k-point lists differ")
    if not MMN_PATH.is_file():
        raise FileNotFoundError(f"{MMN_PATH} is required; the script never alters its .gz copy")
    return tuple(int(value) for value in checkpoint03["mp_grid"])

## Occupied frames

`C03`, `C04` (the two occupied-subspace bases on the ab-initio mesh) and `Q` (the per-k alignment unitary) are left as top-level notebook variables -- inspect them directly.

In [ ]:
def occupied_frames(checkpoint03: dict, checkpoint04: dict) -> tuple:
    """Build occupied frames and their pointwise change of gauge."""
    V03 = np.asarray(checkpoint03["v_matrix"], dtype=np.complex128)
    V04 = np.asarray(checkpoint04["v_matrix"], dtype=np.complex128)
    eigenvalues = read_eigenvalues(TRIAL03 / f"{PREFIX}.eig", int(checkpoint03["num_kpts"]), int(checkpoint03["num_bands"]))

    # Trial 03 contains ten empty Wannier bands. Diagonalize its projected
    # Hamiltonian only on the ab-initio mesh and retain the lowest 38 states.
    H03 = np.einsum("kbi,kb,kbj->kij", V03.conj(), eigenvalues, V03, optimize=True)
    _, eigenvectors03 = np.linalg.eigh(H03)
    C03 = np.einsum("kbi,kiv->kbv", V03, eigenvectors03[:, :, :N_OCC], optimize=True)

    # Trial 04 has J=M=38, so its whole Wannier frame is occupied.
    if int(checkpoint04["num_wann"]) != N_OCC:
        raise ValueError("trial 04 is not the expected J=M=38 checkpoint")
    C04 = V04

    num_kpts = C03.shape[0]
    identity = np.eye(N_OCC)
    Q = np.empty((num_kpts, N_OCC, N_OCC), dtype=np.complex128)
    minimum_principal_singular_value = np.inf
    maximum_projector_frobenius_difference = 0.0
    maximum_projector_entry_difference = 0.0
    maximum_frame_alignment_relative = 0.0
    maximum_C03_orthonormality = 0.0
    maximum_C04_orthonormality = 0.0

    for ik in range(num_kpts):
        cross = C03[ik].conj().T @ C04[ik]
        Q[ik], singular_values = polar_unitary(cross)
        minimum_principal_singular_value = min(minimum_principal_singular_value, float(np.min(singular_values)))
        projector_difference = C03[ik] @ C03[ik].conj().T - C04[ik] @ C04[ik].conj().T
        maximum_projector_frobenius_difference = max(maximum_projector_frobenius_difference, float(np.linalg.norm(projector_difference)))
        maximum_projector_entry_difference = max(maximum_projector_entry_difference, float(np.max(np.abs(projector_difference))))
        maximum_frame_alignment_relative = max(maximum_frame_alignment_relative, float(np.linalg.norm(C04[ik] - C03[ik] @ Q[ik]) / np.linalg.norm(C04[ik])))
        maximum_C03_orthonormality = max(maximum_C03_orthonormality, float(np.max(np.abs(C03[ik].conj().T @ C03[ik] - identity))))
        maximum_C04_orthonormality = max(maximum_C04_orthonormality, float(np.max(np.abs(C04[ik].conj().T @ C04[ik] - identity))))

    metrics = {
        "minimum_principal_overlap_singular_value": minimum_principal_singular_value,
        "maximum_projector_frobenius_difference": maximum_projector_frobenius_difference,
        "maximum_projector_entry_difference": maximum_projector_entry_difference,
        "maximum_frame_alignment_relative_frobenius": maximum_frame_alignment_relative,
        "maximum_C03_orthonormality_entry_error": maximum_C03_orthonormality,
        "maximum_C04_orthonormality_entry_error": maximum_C04_orthonormality,
    }
    return C03, C04, Q, metrics

## Stream the shared MMN into occupied links

In [ ]:
def stream_occupied_links(checkpoint: dict, C03: np.ndarray, C04: np.ndarray):
    """Stream the common MMN and return occupied links and torus topology."""
    stream = _mmn_records(MMN_PATH)
    num_bands, num_kpts, num_neighbors = next(stream)
    for key, value in (("num_bands", num_bands), ("num_kpts", num_kpts), ("nntot", num_neighbors)):
        if int(checkpoint[key]) != value:
            raise ValueError(f"{MMN_PATH}: {key}={value} disagrees with checkpoint {checkpoint[key]}")

    links03 = np.empty((num_kpts, num_neighbors, N_OCC, N_OCC), dtype=np.complex128)
    links04 = np.empty_like(links03)
    neighbors = np.empty((num_kpts, num_neighbors), dtype=int)
    shifts = np.empty((num_kpts, num_neighbors, 3), dtype=int)
    steps = np.empty_like(shifts)
    kpoints = np.asarray(checkpoint["kpt_red"], dtype=float)
    grid = np.asarray(checkpoint["mp_grid"], dtype=int)

    for index, record in enumerate(stream):
        ik, ib = divmod(index, num_neighbors)
        neighbor, shift = _mmn_header(MMN_PATH, record, ik, num_kpts)
        neighbors[ik, ib] = neighbor
        shifts[ik, ib] = shift
        displacement = kpoints[neighbor] + shift - kpoints[ik]
        integer_step = np.rint(displacement * grid).astype(int)
        if not np.allclose(displacement * grid, integer_step, atol=2e-10, rtol=0):
            raise ValueError("an MMN displacement is not a mesh step")
        if np.count_nonzero(integer_step) != 1 or np.max(np.abs(integer_step)) != 1:
            raise ValueError(f"expected a nearest-axis MMN link, got step {integer_step}")
        steps[ik, ib] = integer_step

        body = record[5:].reshape((num_bands, num_bands, 2))
        overlap = (body[..., 0] + 1j * body[..., 1]).T
        links03[ik, ib] = C03[ik].conj().T @ overlap @ C03[neighbor]
        links04[ik, ib] = C04[ik].conj().T @ overlap @ C04[neighbor]

        if (index + 1) % 512 == 0:
            print(f"  streamed {index + 1}/{num_kpts * num_neighbors} MMN records", flush=True)

    return links03, links04, neighbors, shifts, steps


def directional_slots(neighbors: np.ndarray, steps: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Index each k-point's +/- Cartesian link on the periodic torus."""
    num_kpts = neighbors.shape[0]
    slots = np.full((num_kpts, 3, 2), -1, dtype=int)
    destinations = np.full((num_kpts, 3, 2), -1, dtype=int)
    for ik in range(num_kpts):
        for ib, step in enumerate(steps[ik]):
            axis = int(np.flatnonzero(step)[0])
            sign_index = 1 if step[axis] > 0 else 0
            if slots[ik, axis, sign_index] >= 0:
                raise ValueError("duplicate MMN direction at one k point")
            slots[ik, axis, sign_index] = ib
            destinations[ik, axis, sign_index] = neighbors[ik, ib]
    if np.any(slots < 0):
        raise ValueError("the MMN does not contain every +/- Cartesian link")
    return slots, destinations

## Polarize links and compare trials

In [ ]:
def analyze_links(links03, links04, neighbors, slots, destinations, Q):
    """Polarize links and measure gauge-covariant and reversal residuals."""
    unitary03 = np.empty_like(links03)
    unitary04 = np.empty_like(links04)
    aligned_raw_relative = []
    aligned_unitary_relative = []
    singular_value_differences = []
    raw_accumulator_numerator = 0.0
    raw_accumulator_denominator = 0.0
    unitary_accumulator_numerator = 0.0
    unitary_accumulator_denominator = 0.0

    for ik in range(links03.shape[0]):
        for ib in range(links03.shape[1]):
            neighbor = neighbors[ik, ib]
            aligned04 = Q[ik] @ links04[ik, ib] @ Q[neighbor].conj().T
            difference = aligned04 - links03[ik, ib]
            reference_norm = np.linalg.norm(links03[ik, ib])
            aligned_raw_relative.append(float(np.linalg.norm(difference) / reference_norm))
            raw_accumulator_numerator += float(np.linalg.norm(difference) ** 2)
            raw_accumulator_denominator += float(reference_norm**2)

            unitary03[ik, ib], singular03 = polar_unitary(links03[ik, ib])
            unitary04[ik, ib], singular04 = polar_unitary(links04[ik, ib])
            singular_value_differences.append(float(np.max(np.abs(singular04 - singular03))))
            aligned_unitary04 = Q[ik] @ unitary04[ik, ib] @ Q[neighbor].conj().T
            difference_unitary = aligned_unitary04 - unitary03[ik, ib]
            unitary_norm = np.linalg.norm(unitary03[ik, ib])
            aligned_unitary_relative.append(float(np.linalg.norm(difference_unitary) / unitary_norm))
            unitary_accumulator_numerator += float(np.linalg.norm(difference_unitary) ** 2)
            unitary_accumulator_denominator += float(unitary_norm**2)

    reversal = {"trial03": {"raw": [], "unitary": []}, "trial04": {"raw": [], "unitary": []}}
    for ik in range(links03.shape[0]):
        for axis in range(3):
            plus_slot = slots[ik, axis, 1]
            neighbor = destinations[ik, axis, 1]
            minus_slot = slots[neighbor, axis, 0]
            if destinations[neighbor, axis, 0] != ik:
                raise ValueError("positive and negative MMN links do not reverse")
            for name, raw, unitary in (("trial03", links03, unitary03), ("trial04", links04, unitary04)):
                raw_difference = raw[neighbor, minus_slot] - raw[ik, plus_slot].conj().T
                reversal[name]["raw"].append(float(np.linalg.norm(raw_difference) / np.linalg.norm(raw[ik, plus_slot])))
                unitary_difference = unitary[neighbor, minus_slot] - unitary[ik, plus_slot].conj().T
                reversal[name]["unitary"].append(float(np.linalg.norm(unitary_difference) / np.linalg.norm(unitary[ik, plus_slot])))

    metrics = {
        "gauge_aligned_raw_link_relative_frobenius": {
            "maximum": float(np.max(aligned_raw_relative)),
            "rms_over_links": float(np.sqrt(np.mean(np.square(aligned_raw_relative)))),
            "global_l2": float(np.sqrt(raw_accumulator_numerator / raw_accumulator_denominator)),
        },
        "gauge_aligned_polar_link_relative_frobenius": {
            "maximum": float(np.max(aligned_unitary_relative)),
            "rms_over_links": float(np.sqrt(np.mean(np.square(aligned_unitary_relative)))),
            "global_l2": float(np.sqrt(unitary_accumulator_numerator / unitary_accumulator_denominator)),
        },
        "maximum_raw_link_singular_value_difference": float(np.max(singular_value_differences)),
        "link_reversal_relative_frobenius": {
            name: {
                kind: {"maximum": float(np.max(values)), "rms": float(np.sqrt(np.mean(np.square(values))))}
                for kind, values in kinds.items()
            }
            for name, kinds in reversal.items()
        },
    }
    return unitary03, unitary04, metrics

## Wilson plaquettes: xy, yz, zx

In [ ]:
def analyze_plaquettes(checkpoint, links03, links04, unitary03, unitary04, slots, destinations, Q):
    """Compare all xy, yz, and zx Wilson plaquettes on the torus."""
    rows: list[dict] = []
    by_orientation: dict[str, dict[str, list[float]]] = {}
    kpoints = np.asarray(checkpoint["kpt_red"], dtype=float)

    for mu, nu, label in PLAQUETTES:
        values = {k: [] for k in (
            "polar_phase_difference", "raw_phase_difference", "directed_phase_difference",
            "polar_aligned_matrix_relative", "raw_aligned_matrix_relative",
            "polar_trace_difference", "raw_trace_difference",
            "trial03_forward_vs_directed_phase", "trial04_forward_vs_directed_phase",
            "trial03_phase", "trial04_phase",
        )}
        for ik in range(links03.shape[0]):
            slot_mu = slots[ik, mu, 1]
            slot_nu = slots[ik, nu, 1]
            k_mu = destinations[ik, mu, 1]
            k_nu = destinations[ik, nu, 1]
            slot_nu_at_mu = slots[k_mu, nu, 1]
            slot_mu_at_nu = slots[k_nu, mu, 1]
            k_mu_nu_a = destinations[k_mu, nu, 1]
            k_mu_nu_b = destinations[k_nu, mu, 1]
            if k_mu_nu_a != k_mu_nu_b:
                raise ValueError("mesh translations do not commute on the torus")
            k_mu_nu = k_mu_nu_a

            loops: dict[str, dict[str, np.ndarray]] = {"trial03": {}, "trial04": {}}
            for name, raw, polar in (("trial03", links03, unitary03), ("trial04", links04, unitary04)):
                loops[name]["raw"] = (
                    raw[ik, slot_mu] @ raw[k_mu, slot_nu_at_mu]
                    @ raw[k_nu, slot_mu_at_nu].conj().T @ raw[ik, slot_nu].conj().T
                )
                loops[name]["polar"] = (
                    polar[ik, slot_mu] @ polar[k_mu, slot_nu_at_mu]
                    @ polar[k_nu, slot_mu_at_nu].conj().T @ polar[ik, slot_nu].conj().T
                )

                # Independently use the MMN's actual negative-direction links
                # for the last two edges of the same oriented plaquette.
                minus_mu = slots[k_mu_nu, mu, 0]
                minus_nu = slots[k_nu, nu, 0]
                if destinations[k_mu_nu, mu, 0] != k_nu:
                    raise ValueError("-mu plaquette edge has wrong destination")
                if destinations[k_nu, nu, 0] != ik:
                    raise ValueError("-nu plaquette edge has wrong destination")
                loops[name]["directed"] = (
                    polar[ik, slot_mu] @ polar[k_mu, slot_nu_at_mu]
                    @ polar[k_mu_nu, minus_mu] @ polar[k_nu, minus_nu]
                )

            phases = {name: {kind: determinant_phase(matrix) for kind, matrix in kinds.items()} for name, kinds in loops.items()}
            polar_phase_difference = wrapped_phase_difference(phases["trial03"]["polar"], phases["trial04"]["polar"])
            raw_phase_difference = wrapped_phase_difference(phases["trial03"]["raw"], phases["trial04"]["raw"])
            directed_phase_difference = wrapped_phase_difference(phases["trial03"]["directed"], phases["trial04"]["directed"])
            aligned_polar04 = Q[ik] @ loops["trial04"]["polar"] @ Q[ik].conj().T
            aligned_raw04 = Q[ik] @ loops["trial04"]["raw"] @ Q[ik].conj().T
            polar_aligned_relative = float(np.linalg.norm(aligned_polar04 - loops["trial03"]["polar"]) / np.linalg.norm(loops["trial03"]["polar"]))
            raw_aligned_relative = float(np.linalg.norm(aligned_raw04 - loops["trial03"]["raw"]) / np.linalg.norm(loops["trial03"]["raw"]))
            polar_trace_difference = float(abs(np.trace(loops["trial04"]["polar"]) - np.trace(loops["trial03"]["polar"])))
            raw_trace_difference = float(abs(np.trace(loops["trial04"]["raw"]) - np.trace(loops["trial03"]["raw"])))
            forward_directed03 = wrapped_phase_difference(phases["trial03"]["polar"], phases["trial03"]["directed"])
            forward_directed04 = wrapped_phase_difference(phases["trial04"]["polar"], phases["trial04"]["directed"])

            values["polar_phase_difference"].append(polar_phase_difference)
            values["raw_phase_difference"].append(raw_phase_difference)
            values["directed_phase_difference"].append(directed_phase_difference)
            values["polar_aligned_matrix_relative"].append(polar_aligned_relative)
            values["raw_aligned_matrix_relative"].append(raw_aligned_relative)
            values["polar_trace_difference"].append(polar_trace_difference)
            values["raw_trace_difference"].append(raw_trace_difference)
            values["trial03_forward_vs_directed_phase"].append(forward_directed03)
            values["trial04_forward_vs_directed_phase"].append(forward_directed04)
            values["trial03_phase"].append(phases["trial03"]["polar"])
            values["trial04_phase"].append(phases["trial04"]["polar"])

            rows.append({
                "orientation": label, "k_index_zero_based": ik,
                "kx_reduced": float(kpoints[ik, 0]), "ky_reduced": float(kpoints[ik, 1]), "kz_reduced": float(kpoints[ik, 2]),
                "trial03_polar_flux_rad": phases["trial03"]["polar"], "trial04_polar_flux_rad": phases["trial04"]["polar"],
                "trial04_minus_trial03_polar_flux_rad": polar_phase_difference,
                "trial03_raw_flux_rad": phases["trial03"]["raw"], "trial04_raw_flux_rad": phases["trial04"]["raw"],
                "trial04_minus_trial03_raw_flux_rad": raw_phase_difference,
                "trial04_minus_trial03_directed_flux_rad": directed_phase_difference,
                "polar_aligned_loop_relative_frobenius": polar_aligned_relative,
                "raw_aligned_loop_relative_frobenius": raw_aligned_relative,
            })
        by_orientation[label] = values

    metrics: dict[str, dict] = {}
    for label, values in by_orientation.items():
        metrics[label] = {
            "number_of_plaquettes": len(values["polar_phase_difference"]),
            "trial03_maximum_absolute_polar_flux_rad": float(np.max(np.abs(values["trial03_phase"]))),
            "trial04_maximum_absolute_polar_flux_rad": float(np.max(np.abs(values["trial04_phase"]))),
            "polar_flux_difference_rad": {
                "maximum_absolute": float(np.max(np.abs(values["polar_phase_difference"]))),
                "rms": float(np.sqrt(np.mean(np.square(values["polar_phase_difference"])))),
            },
            "raw_flux_difference_rad": {
                "maximum_absolute": float(np.max(np.abs(values["raw_phase_difference"]))),
                "rms": float(np.sqrt(np.mean(np.square(values["raw_phase_difference"])))),
            },
            "directed_polar_flux_difference_rad": {
                "maximum_absolute": float(np.max(np.abs(values["directed_phase_difference"]))),
                "rms": float(np.sqrt(np.mean(np.square(values["directed_phase_difference"])))),
            },
            "gauge_aligned_polar_loop_relative_frobenius": {
                "maximum": float(np.max(values["polar_aligned_matrix_relative"])),
                "rms": float(np.sqrt(np.mean(np.square(values["polar_aligned_matrix_relative"])))),
            },
            "gauge_aligned_raw_loop_relative_frobenius": {
                "maximum": float(np.max(values["raw_aligned_matrix_relative"])),
                "rms": float(np.sqrt(np.mean(np.square(values["raw_aligned_matrix_relative"])))),
            },
            "maximum_absolute_polar_trace_difference": float(np.max(values["polar_trace_difference"])),
            "maximum_absolute_raw_trace_difference": float(np.max(values["raw_trace_difference"])),
            "maximum_forward_dagger_vs_directed_flux_difference_rad": {
                "trial03": float(np.max(np.abs(values["trial03_forward_vs_directed_phase"]))),
                "trial04": float(np.max(np.abs(values["trial04_forward_vs_directed_phase"]))),
            },
        }
    return rows, metrics


def save_results(output: Path, rows: list[dict], summary: dict) -> None:
    """Write the per-plaquette CSV and complete JSON summary."""
    output.mkdir(parents=True, exist_ok=True)
    csv_path = output / "raw_mmn_wilson_plaquettes.csv"
    json_path = output / "raw_mmn_wilson_summary.json"
    with csv_path.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    json_path.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n")
    print(f"saved {csv_path}")
    print(f"saved {json_path}")

## Run each step

Separate cells so `checkpoint03`, `checkpoint04`, `C03`, `C04`, `Q`, `links03`, `links04`, `unitary03`, `unitary04`, and `rows` all stay around as inspectable variables afterward.

In [ ]:
if RUN_COMPUTATION:
    print("reading checkpoints", flush=True)
    checkpoint03 = read_wannier_checkpoint(TRIAL03 / f"{PREFIX}.chk")
    checkpoint04 = read_wannier_checkpoint(TRIAL04 / f"{PREFIX}.chk")
    grid = validate_inputs(checkpoint03, checkpoint04)

In [ ]:
if RUN_COMPUTATION:
    print("constructing the two occupied frames on the ab-initio mesh", flush=True)
    C03, C04, Q, projector_metrics = occupied_frames(checkpoint03, checkpoint04)
    print(f"  max projector Frobenius difference: {projector_metrics['maximum_projector_frobenius_difference']:.3e}")

In [ ]:
if RUN_COMPUTATION:
    print(f"streaming shared MMN: {MMN_PATH}", flush=True)
    links03, links04, neighbors, shifts, steps = stream_occupied_links(checkpoint03, C03, C04)
    slots, destinations = directional_slots(neighbors, steps)

In [ ]:
if RUN_COMPUTATION:
    print("polarizing and comparing all occupied links", flush=True)
    unitary03, unitary04, link_metrics = analyze_links(links03, links04, neighbors, slots, destinations, Q)

In [ ]:
if RUN_COMPUTATION:
    print("checking xy, yz, and zx Wilson plaquettes", flush=True)
    rows, plaquette_metrics = analyze_plaquettes(checkpoint03, links03, links04, unitary03, unitary04, slots, destinations, Q)

## Assemble the summary and save

In [ ]:
if RUN_COMPUTATION:
    summary = {
        "description": (
            "Raw occupied-link and Wilson-plaquette comparison; no A(R), "
            "H(R), derivatives, or off-grid interpolation"
        ),
        "input": {
            "trial03_checkpoint": str(TRIAL03 / f"{PREFIX}.chk"),
            "trial04_checkpoint": str(TRIAL04 / f"{PREFIX}.chk"),
            "shared_mmn": str(MMN_PATH),
            "mesh": list(grid),
            "num_kpts": int(checkpoint03["num_kpts"]),
            "num_mmn_neighbors": int(checkpoint03["nntot"]),
            "num_ab_initio_bands": int(checkpoint03["num_bands"]),
            "num_occupied": N_OCC,
            "trial03_num_wann": int(checkpoint03["num_wann"]),
            "trial04_num_wann": int(checkpoint04["num_wann"]),
        },
        "gauge_handling": {
            "occupied_link": "L_a(k,b) = C_a(k)^dagger M(k,b) C_a(k+b)",
            "frame_alignment": (
                "Q(k) = polar[C_3(k)^dagger C_4(k)]; compare "
                "Q(k)L_4(k,b)Q(k+b)^dagger with L_3(k,b)"
            ),
            "polar_link": "U_a(k,b) = polar[L_a(k,b)]",
            "wilson_loop": "W_mn(k)=U_m(k)U_n(k+m)U_m(k+n)^daggerU_n(k)^dagger",
            "gauge_invariant_flux": "phi_mn(k) = arg det W_mn(k)",
        },
        "projector": projector_metrics,
        "links": link_metrics,
        "plaquettes": plaquette_metrics,
    }
    save_results(OUTPUT_DIR, rows, summary)

## Load a previous run (skip if you just computed above)

In [ ]:
if not RUN_COMPUTATION:
    summary = json.loads((OUTPUT_DIR / "raw_mmn_wilson_summary.json").read_text())
    with (OUTPUT_DIR / "raw_mmn_wilson_plaquettes.csv").open() as f:
        rows = list(csv.DictReader(f))

print(json.dumps(summary, indent=2, sort_keys=True))
print(f"\n{len(rows)} plaquette rows loaded")